# QLOP — AI-Driven Skill Gap Analysis: NER Model for IT Resume Parsing
## Team CC26-PSU101 | Capstone Project

**Objective:** Fine-tune `yashpwr/resume-ner-bert-v2` (PyTorch → TensorFlow) for extracting  
hard IT skills and professional entities from resumes using a fully custom TF/Keras pipeline.

| Requirement | Status |
|---|---|
| TF Porting (`from_pt=True`) | ✅ |
| Functional API / Subclassing | ✅ |
| Heavy Fine-Tuning (unfrozen BERT) | ✅ |
| Custom Loss Function | ✅ |
| Custom Callback | ✅ |
| Custom Training Loop (`tf.GradientTape`) | ✅ |
| TensorBoard Integration | ✅ |
| Evaluation Metrics (F1 ≥ 85%) | ✅ |
| Model Export (`.keras` / SavedModel) | ✅ |
| Inference Script | ✅ |


## Install & Import Dependencies

In [ ]:
# ── Install dependencies
import subprocess
import sys

packages = [
    "transformers==4.40.2",
    "datasets",
    "seqeval",
    "tensorboard",
    "scikit-learn",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + packages,
    check=True
)

print("All packages installed successfully.")

In [ ]:
# ── Core imports
import os
import json
import time
import datetime
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import tensorflow as tf

from transformers import (
    AutoTokenizer,
    TFBertForTokenClassification,
    DataCollatorForTokenClassification,
)

from datasets import load_dataset, DatasetDict, Dataset

from seqeval.metrics import (
    classification_report,
    f1_score as seq_f1,
    precision_score as seq_precision,
    recall_score as seq_recall,
)

from sklearn.metrics import accuracy_score

print(f"TensorFlow : {tf.__version__}")
print(f"GPU count  : {len(tf.config.list_physical_devices('GPU'))}")

In [ ]:
import tensorflow as tf

print(tf.config.list_physical_devices('GPU'))

## Multi-GPU Strategy (Kaggle T4 x2)

In [ ]:
# ── MirroredStrategy for Kaggle dual T4
gpus = tf.config.list_physical_devices('GPU')
if len(gpus) >= 2:
    strategy = tf.distribute.MirroredStrategy()
    print(f"MirroredStrategy active — using {strategy.num_replicas_in_sync} GPUs")
elif len(gpus) == 1:
    strategy = tf.distribute.OneDeviceStrategy("/GPU:0")
    print("Single GPU strategy")
else:
    strategy = tf.distribute.OneDeviceStrategy("/CPU:0")
    print("No GPU found — using CPU")

NUM_REPLICAS = strategy.num_replicas_in_sync


## Global Configuration

In [ ]:
# ── Hyperparameters & paths 
CFG = {
    # Model
    "base_model"      : "yashpwr/resume-ner-bert-v2",
    "max_length"      : 128,

    # Training
    "epochs"          : 5,
    "batch_size"      : 16 * NUM_REPLICAS,   # scale with GPU count
    "learning_rate"   : 2e-5,
    "warmup_ratio"    : 0.1,
    "weight_decay"    : 0.01,
    "dropout_rate"    : 0.1,

    # Data
    "train_size"      : 0.8,
    "val_size"        : 0.1,
    "test_size"       : 0.1,
    "max_samples"     : 2000,   # cap for fast iteration; set None for full dataset

    # Paths
    "log_dir"         : "/kaggle/working/logs",
    "model_save_path" : "/kaggle/working/qlop_ner_model",
    "keras_save_path" : "/kaggle/working/qlop_ner_model.keras",

    # IT Hard-Skill focus label set (subset of the 25 BIO labels we care most about)
    "it_skill_labels" : ["Skills", "Designation", "Companies worked at",
                         "Degree", "College Name", "Years of Experience"],
}

os.makedirs(CFG["log_dir"], exist_ok=True)
os.makedirs(CFG["model_save_path"], exist_ok=True)
print("Config ready:", json.dumps({k: v for k, v in CFG.items() if k != "it_skill_labels"}, indent=2))


## Label Schema (BIO Tags — IT Focus)

In [ ]:
# ── BIO label schema matching yashpwr/resume-ner-bert-v2 ─────────────────────
# The model uses 25 entity types encoded in BIO scheme.
# We preserve ALL labels so fine-tuning is consistent with the pre-trained head.

RAW_LABELS = [
    "Name", "Email Address", "Phone", "Location",
    "Companies worked at", "Designation", "Skills",
    "Years of Experience", "Degree", "College Name",
    "Graduation Year", "UNKNOWN",
]

# Build full BIO label list
BIO_LABELS = ["O"]
for lbl in RAW_LABELS:
    BIO_LABELS.append(f"B-{lbl}")
    BIO_LABELS.append(f"I-{lbl}")

# The pre-trained model has its own id2label; we'll ALIGN to it after loading.
# For data preprocessing we use this schema as fallback.
LABEL2ID = {lbl: i for i, lbl in enumerate(BIO_LABELS)}
ID2LABEL = {i: lbl for lbl, i in LABEL2ID.items()}
NUM_LABELS_LOCAL = len(BIO_LABELS)

print(f"Local BIO label count : {NUM_LABELS_LOCAL}")
print("Labels:", BIO_LABELS[:10], "...")

# ── IT Hard-Skill entity types we prioritise 
IT_SKILL_ENTITY_TYPES = {
    "Skills", "Designation", "Companies worked at",
    "Degree", "College Name", "Years of Experience",
}
print("\nIT-focused entity types:", IT_SKILL_ENTITY_TYPES)


## Data Loading & Preprocessing

We use the **`dataturks-resume-ner`** dataset from Hugging Face.  
If the remote dataset is unavailable we fall back to a **programmatically generated**  
synthetic dataset that covers all 12 IT-resume entity types above.


In [ ]:
# ── 5.1 Load REAL Resume NER Dataset 
import json
import spacy
import pandas as pd
from datasets import Dataset, DatasetDict

def load_dataturks_from_kaggle(filepath="/kaggle/input/datasets/dataturks/resume-entities-for-ner/Entity Recognition in Resumes.json"):
    print("Memproses dataset asli dari file JSON DataTurks...")
    # Menggunakan tokenizer super cepat bawaan Spacy
    nlp = spacy.blank("en") 
    
    parsed_data = []
    
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                item = json.loads(line)
                text = item['content']
                annotations = item.get('annotation', [])
                
                spans = []
                if annotations:
                    for ann in annotations:
                        label = ann['label'][0]
                        # Dataturks menggunakan format offset karakter
                        start = ann['points'][0]['start']
                        end = ann['points'][0]['end'] + 1 
                        
                        span = nlp.make_doc(text).char_span(start, end, label=label)
                        if span is not None:
                            spans.append(span)
                
                # Filter untuk membuang anomali entitas yang saling tumpang tindih
                filtered_spans = spacy.util.filter_spans(spans)
                valid_entities = [(s.start_char, s.end_char, s.label_) for s in filtered_spans]
                
                doc = nlp(text)
                tags = spacy.training.iob_utils.offsets_to_biluo_tags(doc, valid_entities)
                
                tokens = []
                ner_tags_int = []
                
                for token, tag in zip(doc, tags):
                    tokens.append(token.text)
                    
                    # Konversi format BILUO dari Spacy ke format BIO
                    if tag.startswith("U-"): bio_tag = "B-" + tag[2:]
                    elif tag.startswith("L-"): bio_tag = "I-" + tag[2:]
                    elif tag == "-" or tag == "O": bio_tag = "O"
                    else: bio_tag = tag
                    
                    # Konversi string tag menjadi ID integer berdasarkan LABEL2ID dari Section 4
                    ner_tags_int.append(LABEL2ID.get(bio_tag, 0))
                    
                parsed_data.append({
                    "tokens": tokens,
                    "ner_tags": ner_tags_int
                })
            except Exception as e:
                continue
                
    print(f"Berhasil mengekstrak {len(parsed_data)} CV asli menjadi token!")
    
    # Ubah ke HuggingFace Dataset dan lakukan Splitting (Train 80, Val 10, Test 10)
    df = pd.DataFrame(parsed_data)
    dataset = Dataset.from_pandas(df)
    
    train_test = dataset.train_test_split(test_size=0.2, seed=42)
    valid_test = train_test['test'].train_test_split(test_size=0.5, seed=42)
    
    return DatasetDict({
        "train": train_test['train'],
        "validation": valid_test['train'],
        "test": valid_test['test']
    })

# Panggil fungsinya
raw_datasets = load_dataturks_from_kaggle()
USING_SYNTHETIC = False

print("\nDistribusi Dataset (Real Data):")
for split, ds_split in raw_datasets.items():
    print(f"  {split:12s}: {len(ds_split):,} sampel")

In [ ]:
# # ── 5.1 Load REAL Resume NER Dataset ───────────────────────────────────────────
# from datasets import load_dataset, DatasetDict

# def load_real_resume_dataset():
#     print("Mengunduh dataset CV asli (BIO tags) dari Hugging Face...")
#     # Menggunakan dataset asli dari yashpwr yang formatnya sudah parquet
#     ds = load_dataset("yashpwr/resume-ner-training-data")
    
#     print("Dataset asli berhasil dimuat!")
    
#     # Dataset ini hanya memiliki split 'train'. Kita pecah menjadi train/val/test
#     train_test = ds['train'].train_test_split(test_size=0.2, seed=42)
#     valid_test = train_test['test'].train_test_split(test_size=0.5, seed=42)
    
#     final_ds = DatasetDict({
#         "train": train_test["train"],
#         "validation": valid_test["train"],
#         "test": valid_test["test"],
#     })
    
#     # Standardisasi nama kolom (sangat penting untuk Section 8)
#     if "words" in final_ds["train"].column_names:
#         final_ds = final_ds.rename_column("words", "tokens")
#     if "labels" in final_ds["train"].column_names:
#         final_ds = final_ds.rename_column("labels", "ner_tags")
        
#     return final_ds

# raw_datasets = load_real_resume_dataset()
# USING_SYNTHETIC = False

# print("\nDistribusi Dataset (Real Data):")
# for split, ds_split in raw_datasets.items():
#     print(f"  {split:12s}: {len(ds_split):,} sampel")

In [ ]:
# def generate_synthetic_dataset(n_samples: int = 2000) -> DatasetDict:
#     """
#     Rule-based synthetic resume NER dataset.
#     Mimics the structure of DataTurks: {'tokens': [...], 'ner_tags': [...]}.
#     """
#     import random
#     random.seed(42)

#     NAMES       = ["Alice Chen", "Bob Kumar", "Carlos Silva", "Diana Lee", "Evan Park"]
#     COMPANIES   = ["Google", "Microsoft", "Amazon", "Meta", "Infosys", "TCS", "Oracle"]
#     DESIGNATIONS= ["Software Engineer", "Data Scientist", "ML Engineer",
#                    "Backend Developer", "DevOps Engineer", "Cloud Architect",
#                    "Full Stack Developer", "Data Analyst"]
#     SKILLS_POOL = [
#         "Python", "Java", "JavaScript", "TypeScript", "Go", "Rust", "C++",
#         "TensorFlow", "PyTorch", "Keras", "Scikit-learn", "Spark", "Hadoop",
#         "React", "Node.js", "FastAPI", "Django", "Flask", "Spring Boot",
#         "Docker", "Kubernetes", "AWS", "GCP", "Azure", "Terraform",
#         "MySQL", "PostgreSQL", "MongoDB", "Redis", "Kafka",
#         "Git", "CI/CD", "Linux", "REST API", "GraphQL",
#     ]
#     DEGREES     = ["B.Tech in Computer Science", "M.Tech in AI", "B.Sc Computer Science",
#                    "M.Sc Data Science", "B.E. Information Technology"]
#     COLLEGES    = ["IIT Delhi", "NIT Trichy", "Stanford University",
#                    "MIT", "UC Berkeley", "VIT University"]
#     GRAD_YEARS  = ["2018", "2019", "2020", "2021", "2022", "2023"]
#     LOCATIONS   = ["Bangalore", "Mumbai", "Delhi", "San Francisco", "New York"]
#     EXPERIENCES = ["3 years", "5 years", "7+ years", "10 years", "2 years"]

#     def _tag(tokens: list, label: str) -> list:
#         """Return BIO tags for a token sequence."""
#         if not tokens:
#             return []
#         tags = [f"B-{label}"] + [f"I-{label}"] * (len(tokens) - 1)
#         return tags

#     samples = []
#     for _ in range(n_samples):
#         tokens, tags = [], []

#         # Name
#         name = random.choice(NAMES)
#         toks = name.split()
#         tokens += toks; tags += _tag(toks, "Name")
#         tokens += ["is", "a"]; tags += ["O", "O"]

#         # Designation
#         desig = random.choice(DESIGNATIONS)
#         toks = desig.split()
#         tokens += toks; tags += _tag(toks, "Designation")

#         # Experience
#         exp = random.choice(EXPERIENCES)
#         tokens += ["with"]; tags += ["O"]
#         toks = exp.split()
#         tokens += toks; tags += _tag(toks, "Years of Experience")
#         tokens += ["of", "experience", "at"]; tags += ["O", "O", "O"]

#         # Company
#         company = random.choice(COMPANIES)
#         toks = company.split()
#         tokens += toks; tags += _tag(toks, "Companies worked at")
#         tokens += ["."]; tags += ["O"]

#         # Skills (2-5 skills)
#         n_skills = random.randint(2, 5)
#         skill_sample = random.sample(SKILLS_POOL, n_skills)
#         tokens += ["Skills", ":"]; tags += ["O", "O"]
#         for idx, sk in enumerate(skill_sample):
#             toks = sk.split()
#             tokens += toks; tags += _tag(toks, "Skills")
#             if idx < n_skills - 1:
#                 tokens += [","]; tags += ["O"]

#         # Degree
#         degree = random.choice(DEGREES)
#         tokens += ["."]; tags += ["O"]
#         toks = degree.split()
#         tokens += toks; tags += _tag(toks, "Degree")
#         tokens += ["from"]; tags += ["O"]

#         # College
#         college = random.choice(COLLEGES)
#         toks = college.split()
#         tokens += toks; tags += _tag(toks, "College Name")

#         # Grad year
#         year = random.choice(GRAD_YEARS)
#         tokens += ["(", year, ")"]; tags += ["O"] + _tag([year], "Graduation Year") + ["O"]

#         # Location & contact
#         loc = random.choice(LOCATIONS)
#         tokens += ["Location", ":"]; tags += ["O", "O"]
#         toks = loc.split()
#         tokens += toks; tags += _tag(toks, "Location")

#         assert len(tokens) == len(tags), "Token/tag length mismatch!"
#         samples.append({"tokens": tokens, "ner_tags": [LABEL2ID.get(t, 0) for t in tags]})

#     # Shuffle and split
#     random.shuffle(samples)
#     n_train = int(0.8 * n_samples)
#     n_val   = int(0.1 * n_samples)

#     def _to_ds(lst):
#         return Dataset.from_dict({
#             "tokens"  : [s["tokens"]   for s in lst],
#             "ner_tags": [s["ner_tags"] for s in lst],
#         })

#     return DatasetDict({
#         "train"     : _to_ds(samples[:n_train]),
#         "validation": _to_ds(samples[n_train:n_train + n_val]),
#         "test"      : _to_ds(samples[n_train + n_val:]),
#     })


# raw_datasets = load_dataturks_dataset()
# if raw_datasets is None:
#     raw_datasets = generate_synthetic_dataset(CFG["max_samples"] or 2000)
#     USING_SYNTHETIC = True
# else:
#     USING_SYNTHETIC = False

# print("\nDataset splits:")
# for split, ds in raw_datasets.items():
#     print(f"  {split:12s}: {len(ds):,} samples")


## Exploratory Data Analysis (EDA)

In [ ]:
import collections, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

train_ds = raw_datasets["train"]

# ── Token & sequence length distribution 
seq_lengths = [len(s) for s in train_ds["tokens"]]
print(f"Sequence length — mean: {np.mean(seq_lengths):.1f}  "
      f"median: {np.median(seq_lengths):.1f}  "
      f"max: {np.max(seq_lengths)}  "
      f"min: {np.min(seq_lengths)}")

# ── Entity-type frequency 
entity_counts = collections.Counter()
for tags in train_ds["ner_tags"]:
    for t in tags:
        label = ID2LABEL.get(t, "O")
        if label != "O" and label.startswith("B-"):
            entity_counts[label[2:]] += 1

print("\nTop entity counts (train split):")
for lbl, cnt in entity_counts.most_common(15):
    bar = "█" * (cnt // max(1, max(entity_counts.values()) // 40))
    print(f"  {lbl:30s} {cnt:6d}  {bar}")

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(seq_lengths, bins=30, color="#4A90D9", edgecolor="white")
axes[0].set_title("Sequence Length Distribution (train)")
axes[0].set_xlabel("# tokens"); axes[0].set_ylabel("count")
axes[0].axvline(CFG["max_length"], color="red", ls="--", label=f"max_length={CFG['max_length']}")
axes[0].legend()

labels_sorted = sorted(entity_counts, key=entity_counts.get, reverse=True)[:12]
counts_sorted = [entity_counts[l] for l in labels_sorted]
colors = ["#E8453C" if l in IT_SKILL_ENTITY_TYPES else "#4A90D9" for l in labels_sorted]
axes[1].barh(labels_sorted[::-1], counts_sorted[::-1], color=colors[::-1])
axes[1].set_title("Entity Frequency (🔴 = IT-focus)")
axes[1].set_xlabel("count")

plt.tight_layout()
plt.savefig("/kaggle/working/eda_plots.png", dpi=120, bbox_inches="tight")
plt.show()
print("EDA plots saved.")


## Tokenizer & Label Alignment

In [ ]:
from transformers import AutoTokenizer, AutoConfig

# ── Load tokenizer from the pre-trained model hub 
tokenizer = AutoTokenizer.from_pretrained(CFG["base_model"])
print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")
print(f"   Vocab size   : {tokenizer.vocab_size:,}")
print(f"   Max length   : {tokenizer.model_max_length}")

# ── Align label schema with pre-trained model
# Kita gunakan AutoConfig alih-alih meload seluruh model.
# Ini jauh lebih cepat, hemat memori, dan mencegah error file .bin!
config = AutoConfig.from_pretrained(CFG["base_model"])

PRETRAINED_ID2LABEL = config.id2label
PRETRAINED_LABEL2ID = {v: k for k, v in PRETRAINED_ID2LABEL.items()}
NUM_LABELS = len(PRETRAINED_ID2LABEL)

print(f"\nPre-trained model label count : {NUM_LABELS}")
print("Sample id2label:", dict(list(PRETRAINED_ID2LABEL.items())[:6]))

## Data Preprocessing & tf.data Pipeline

In [ ]:
# ── 8.1  Tokenize & align labels ─────────────────────────────────────────────
# BERT sub-word tokenization splits tokens into pieces; we need to propagate
# the NER tag to the FIRST sub-token and set -100 (ignore_index) for the rest.

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=CFG["max_length"],
        padding="max_length",
        is_split_into_words=True,
        return_offsets_mapping=False,
    )

    all_labels = []
    for i, ner_tags in enumerate(examples["ner_tags"]):
        word_ids  = tokenized.word_ids(batch_index=i)
        prev_word = None
        label_ids = []
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)          # [CLS], [SEP], PAD
            elif word_id != prev_word:
                # First sub-token of the word → use the original tag
                raw_tag = ner_tags[word_id] if word_id < len(ner_tags) else 0
                # Remap from local LABEL2ID to pre-trained LABEL2ID
                local_label_str = ID2LABEL.get(raw_tag, "O")
                # Try direct match, else "O"
                remapped = PRETRAINED_LABEL2ID.get(local_label_str,
                           PRETRAINED_LABEL2ID.get("O", 0))
                label_ids.append(remapped)
            else:
                # Continuation sub-token → ignore
                label_ids.append(-100)
            prev_word = word_id
        all_labels.append(label_ids)

    tokenized["labels"] = all_labels
    return tokenized


print("Tokenising dataset…")
tokenized_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)
print("Tokenisation done.")
print("Features:", list(tokenized_datasets["train"].features.keys()))


In [ ]:
# ── 8.2  Convert to tf.data.Dataset

IGNORE_INDEX = -100   # labels with this value are masked in loss & metrics

def hf_to_tf_dataset(hf_ds, batch_size: int, shuffle: bool = False) -> tf.data.Dataset:
    """Convert a HuggingFace Dataset to a tf.data.Dataset of (inputs, labels) tuples."""

    def gen():
        for sample in hf_ds:
            yield (
                {
                    "input_ids"     : sample["input_ids"],
                    "attention_mask": sample["attention_mask"],
                    "token_type_ids": sample.get("token_type_ids",
                                                  [0] * CFG["max_length"]),
                },
                sample["labels"],
            )

    output_sig = (
        {
            "input_ids"     : tf.TensorSpec([CFG["max_length"]], tf.int32),
            "attention_mask": tf.TensorSpec([CFG["max_length"]], tf.int32),
            "token_type_ids": tf.TensorSpec([CFG["max_length"]], tf.int32),
        },
        tf.TensorSpec([CFG["max_length"]], tf.int32),
    )

    ds = tf.data.Dataset.from_generator(gen, output_signature=output_sig)
    
    # ── BARIS TAMBAHAN: Paksa TF mengetahui panjang dataset ──
    ds = ds.apply(tf.data.experimental.assert_cardinality(len(hf_ds)))
    
    if shuffle:
        ds = ds.shuffle(buffer_size=1000, seed=42)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


BATCH = CFG["batch_size"]
train_tf = hf_to_tf_dataset(tokenized_datasets["train"],      BATCH, shuffle=True)
val_tf   = hf_to_tf_dataset(tokenized_datasets["validation"], BATCH, shuffle=False)
test_tf  = hf_to_tf_dataset(tokenized_datasets["test"],       BATCH, shuffle=False)

print(f"tf.data pipelines ready")
print(f"   train batches : {len(train_tf)}")
print(f"   val   batches : {len(val_tf)}")
print(f"   test  batches : {len(test_tf)}")


## Section 9 — Model Architecture  
### TF Porting + Functional API Wrapper + Custom Layers

We:
1. Load `yashpwr/resume-ner-bert-v2` **from PyTorch weights** via `from_pt=True`
2. Wrap it in a `tf.keras.Model` subclass that adds:
   - A **custom Dropout layer** for regularisation
   - An optional **IT-skill attention re-scoring head** (custom dense projection)
3. Unfreeze the **top 4 BERT encoder layers + pooler** for heavy fine-tuning


In [ ]:
# ── 9.1  Custom IT-Skill Re-Scoring Head (Custom Layer) 
import tensorflow as tf

class ITSkillAttentionHead(tf.keras.layers.Layer):
    """
    A lightweight residual projection that re-scores token logits
    to amplify IT hard-skill entities.
    Architecture: Dense(hidden) → GELU → Dense(num_labels) → residual add
    """
    def __init__(self, num_labels: int, hidden_dim: int = 256, dropout_rate: float = 0.1, **kw):
        super().__init__(**kw)
        self.num_labels    = num_labels
        self.dense1        = tf.keras.layers.Dense(hidden_dim, activation="gelu",
                                                   name="it_dense1")
        self.dropout       = tf.keras.layers.Dropout(dropout_rate, name="it_dropout")
        self.dense2        = tf.keras.layers.Dense(num_labels, use_bias=True,
                                                   name="it_dense2")
        # Initialise residual projection near-zero so training starts stable
        self.residual_proj = tf.keras.layers.Dense(num_labels, use_bias=False,
                                                   kernel_initializer="zeros",
                                                   name="it_residual")

    def call(self, logits, training=False):
        # logits: (batch, seq_len, num_labels)
        x = self.dense1(logits)
        x = self.dropout(x, training=training)
        x = self.dense2(x)
        # Residual connection keeps original logits dominant early in training
        return logits + self.residual_proj(logits) + x * 0.1

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"num_labels": self.num_labels,
                    "hidden_dim": self.dense1.units,
                    "dropout_rate": self.dropout.rate})
        return cfg

print("Custom Layer 'ITSkillAttentionHead' berhasil dimuat!")

In [ ]:
import torch
from transformers import BertForTokenClassification

print("Menerjemahkan format Safetensors PyTorch ke TensorFlow...")

# 1. Unduh model aslinya menggunakan PyTorch terlebih dahulu
pt_model = BertForTokenClassification.from_pretrained(
    CFG["base_model"], 
    num_labels=NUM_LABELS, 
    ignore_mismatched_sizes=True
)

# 2. Simpan ke folder lokal Kaggle secara paksa menggunakan format lama (.bin)
LOCAL_MODEL_DIR = "./temp_pt_model"
pt_model.save_pretrained(LOCAL_MODEL_DIR, safe_serialization=False)
print("File terjemahan (pytorch_model.bin) berhasil dibuat secara lokal!")

# 3. Bersihkan memori PyTorch agar RAM/GPU tidak penuh saat TensorFlow berjalan
del pt_model
torch.cuda.empty_cache()

# ── Instantiate inside strategy scope
with strategy.scope():
    # 4. Sekarang, suruh TensorFlow membaca dari folder lokal kita, bukan dari internet!
    _bert_base = TFBertForTokenClassification.from_pretrained(
        LOCAL_MODEL_DIR,
        from_pt=True,                     
        num_labels=NUM_LABELS
    )
    print(f"PyTorch → TF porting complete (num_labels={NUM_LABELS})")

    # 5. Bungkus ke dalam arsitektur Custom Model kita
    model = QLOPNERModel(
        bert_model              = _bert_base,
        num_labels              = NUM_LABELS,
        dropout_rate            = CFG["dropout_rate"],
        unfreeze_last_n_layers  = 4,
        name                    = "qlop_ner",
    )

In [ ]:
# # ── 9.2  QLOP NER Model (Model Subclassing) 

# class QLOPNERModel(keras.Model):
#     """
#     Wraps TFBertForTokenClassification inside a Keras Model (Subclassing API).
#     Adds:
#       • Extra Dropout before the BERT classifier head
#       • ITSkillAttentionHead for residual re-scoring
#     Fine-tuning: top N BERT encoder layers are unfrozen.
#     """

#     def __init__(self,
#                  bert_model: TFBertForTokenClassification,
#                  num_labels: int,
#                  dropout_rate: float = 0.1,
#                  unfreeze_last_n_layers: int = 4,
#                  **kwargs):
#         super().__init__(**kwargs)

#         self.bert_ner      = bert_model
#         self.extra_dropout = keras.layers.Dropout(dropout_rate, name="extra_dropout")
#         self.it_head       = ITSkillAttentionHead(num_labels, hidden_dim=256,
#                                                   dropout_rate=dropout_rate,
#                                                   name="it_skill_head")
#         self.num_labels    = num_labels

#         # ── Freeze backbone, then selectively unfreeze top layers ────────────
#         self.bert_ner.trainable = True   # start fully trainable …

#         # Freeze embedding layer
#         self.bert_ner.bert.embeddings.trainable = False

#         # Freeze encoder layers except the last N
#         n_total = len(self.bert_ner.bert.encoder.layer)
#         for i, layer in enumerate(self.bert_ner.bert.encoder.layer):
#             layer.trainable = (i >= n_total - unfreeze_last_n_layers)

#         trainable   = sum(np.prod(v.shape) for v in self.trainable_variables)
#         untrainable = sum(np.prod(v.shape) for v in self.non_trainable_variables)
#         print(f"Model built — trainable params : {trainable:,} / "
#               f"total : {trainable + untrainable:,}")
#         print(f"   BERT encoder layers unfrozen  : {unfreeze_last_n_layers} / {n_total}")

#     def call(self, inputs, training=False):
#         # inputs: dict with input_ids, attention_mask, token_type_ids
#         bert_out = self.bert_ner(
#             input_ids      = inputs["input_ids"],
#             attention_mask = inputs["attention_mask"],
#             token_type_ids = inputs["token_type_ids"],
#             training       = training,
#         )
#         logits  = bert_out.logits                          # (B, seq, num_labels)
#         logits  = self.extra_dropout(logits, training=training)
#         logits  = self.it_head(logits, training=training)
#         return logits   # return raw logits; loss fn handles masking


# # ── Instantiate inside strategy scope
# with strategy.scope():
#     _bert_base = TFBertForTokenClassification.from_pretrained(
#         CFG["base_model"],
#         from_pt=True,                       # ← PyTorch → TensorFlow porting
#         num_labels=NUM_LABELS,
#         ignore_mismatched_sizes=True,
#     )
#     print(f"PyTorch → TF porting complete  (num_labels={NUM_LABELS})")

#     model = QLOPNERModel(
#         bert_model              = _bert_base,
#         num_labels              = NUM_LABELS,
#         dropout_rate            = CFG["dropout_rate"],
#         unfreeze_last_n_layers  = 4,
#         name                    = "qlop_ner",
#     )


## Custom Components  
### (A) Masked Categorical Cross-Entropy Loss  
### (B) Custom Training Callback


In [ ]:
# ── SECTION 10: Custom Loss & Custom Callback ─────────────────────────────
import tensorflow as tf
import numpy as np
import time

# ── (A) Custom Loss: Masked Sparse Categorical Cross-Entropy
# Padding and sub-word continuation tokens are labelled -100 and MUST be ignored.

# class MaskedSparseCCELoss(tf.keras.losses.Loss):
#     """
#     Sparse Categorical Cross-Entropy that ignores positions with label == ignore_index.
#     Supports class-weighted loss to upweight IT hard-skill entities.
#     """

#     def __init__(self,
#                  ignore_index      : int   = -100,
#                  it_skill_label_ids: list  = None,
#                  skill_weight      : float = 2.0,
#                  label_smoothing   : float = 0.0,
#                  name              : str   = "masked_scce",
#                  **kwargs):
#         super().__init__(name=name, **kwargs)
#         self.ignore_index       = ignore_index
#         self.it_skill_label_ids = it_skill_label_ids or []
#         self.skill_weight       = skill_weight
#         self.label_smoothing    = label_smoothing

#     def call(self, y_true, y_pred):
#         """
#         y_true : (batch, seq_len)  int32  — label ids, -100 = ignore
#         y_pred : (batch, seq_len, num_labels)  float32 — logits
#         """
#         # Mask padding / sub-word continuations
#         mask    = tf.cast(tf.not_equal(y_true, self.ignore_index), tf.float32)

#         # Replace -100 with 0 so gather doesn't error; mask will zero it out
#         y_safe  = tf.where(tf.equal(y_true, self.ignore_index),
#                            tf.zeros_like(y_true), y_true)

#         # Per-token cross-entropy
#         per_tok = tf.keras.losses.sparse_categorical_crossentropy( # ← PERBAIKAN: Menggunakan tf.keras
#             y_safe, y_pred, from_logits=True,
#             label_smoothing=self.label_smoothing
#         )   # (batch, seq_len)

#         # Optional skill upweighting
#         if self.it_skill_label_ids:
#             it_ids  = tf.constant(self.it_skill_label_ids, dtype=tf.int32)
#             is_skill = tf.reduce_any(
#                 tf.equal(tf.expand_dims(y_safe, -1),
#                          tf.reshape(it_ids, [1, 1, -1])),
#                 axis=-1
#             )   # (batch, seq_len)
#             weights = tf.where(is_skill,
#                                tf.fill(tf.shape(mask), self.skill_weight),
#                                tf.ones_like(mask))
#             per_tok = per_tok * weights

#         # Apply mask and compute mean over valid tokens
#         loss = tf.reduce_sum(per_tok * mask) / (tf.reduce_sum(mask) + 1e-8)
#         return loss

#     def get_config(self):
#         cfg = super().get_config()
#         cfg.update({
#             "ignore_index"      : self.ignore_index,
#             "it_skill_label_ids": self.it_skill_label_ids,
#             "skill_weight"      : self.skill_weight,
#             "label_smoothing"   : self.label_smoothing,
#         })
#         return cfg

# ── (A) Custom Loss: Masked Sparse Categorical Cross-Entropy
# Padding and sub-word continuation tokens are labelled -100 and MUST be ignored.

# ── (A) Custom Loss: Masked Sparse Categorical Cross-Entropy
# Padding and sub-word continuation tokens are labelled -100 and MUST be ignored.

# class MaskedSparseCCELoss(tf.keras.losses.Loss):
#     """
#     Categorical Cross-Entropy that ignores positions with label == ignore_index.
#     Supports class-weighted loss to upweight IT hard-skill entities.
#     """

#     def __init__(self,
#                  ignore_index      : int   = -100,
#                  it_skill_label_ids: list  = None,
#                  skill_weight      : float = 2.0,
#                  label_smoothing   : float = 0.0,
#                  name              : str   = "masked_scce",
#                  **kwargs):
#         # ── PERBAIKAN: Matikan reduksi otomatis Keras di sini ──
#         super().__init__(name=name, reduction=tf.keras.losses.Reduction.NONE, **kwargs)
        
#         self.ignore_index       = ignore_index
#         self.it_skill_label_ids = it_skill_label_ids or []
#         self.skill_weight       = skill_weight
#         self.label_smoothing    = label_smoothing

#     def call(self, y_true, y_pred):
#         """
#         y_true : (batch, seq_len)  int32  — label ids, -100 = ignore
#         y_pred : (batch, seq_len, num_labels)  float32 — logits
#         """
#         # Mask padding / sub-word continuations
#         mask    = tf.cast(tf.not_equal(y_true, self.ignore_index), tf.float32)

#         # Replace -100 with 0 so one_hot doesn't error; mask will zero it out later
#         y_safe  = tf.where(tf.equal(y_true, self.ignore_index),
#                            tf.zeros_like(y_true), y_true)

#         # Konversi ke One-Hot agar mendukung label_smoothing
#         num_classes = tf.shape(y_pred)[-1]
#         y_safe_one_hot = tf.one_hot(y_safe, depth=num_classes)

#         # Gunakan categorical_crossentropy standar
#         per_tok = tf.keras.losses.categorical_crossentropy(
#             y_safe_one_hot, y_pred, from_logits=True,
#             label_smoothing=self.label_smoothing
#         )   # (batch, seq_len)

#         # Optional skill upweighting
#         if self.it_skill_label_ids:
#             it_ids  = tf.constant(self.it_skill_label_ids, dtype=tf.int32)
#             is_skill = tf.reduce_any(
#                 tf.equal(tf.expand_dims(y_safe, -1),
#                          tf.reshape(it_ids, [1, 1, -1])),
#                 axis=-1
#             )   # (batch, seq_len)
#             weights = tf.where(is_skill,
#                                tf.fill(tf.shape(mask), self.skill_weight),
#                                tf.ones_like(mask))
#             per_tok = per_tok * weights

#         # Apply mask and compute mean over valid tokens
#         loss = tf.reduce_sum(per_tok * mask) / (tf.reduce_sum(mask) + 1e-8)
#         return loss

#     def get_config(self):
#         cfg = super().get_config()
#         cfg.update({
#             "ignore_index"      : self.ignore_index,
#             "it_skill_label_ids": self.it_skill_label_ids,
#             "skill_weight"      : self.skill_weight,
#             "label_smoothing"   : self.label_smoothing,
#         })
#         return cfg

# # (Pastikan loss_fn di-generate ulang)
# loss_fn = MaskedSparseCCELoss(
#     ignore_index       = IGNORE_INDEX,
#     it_skill_label_ids = IT_SKILL_LABEL_IDS,
#     skill_weight       = 2.0,
#     label_smoothing    = 0.05,
# )
# ── (A) Custom Loss: Masked Sparse Categorical Cross-Entropy
# Padding and sub-word continuation tokens are labelled -100 and MUST be ignored.

class MaskedSparseCCELoss(tf.keras.losses.Loss):
    """
    Categorical Cross-Entropy that ignores positions with label == ignore_index.
    Supports class-weighted loss to upweight IT hard-skill entities.
    """

    def __init__(self,
                 ignore_index      : int   = -100,
                 it_skill_label_ids: list  = None,
                 skill_weight      : float = 2.0,
                 label_smoothing   : float = 0.0,
                 name              : str   = "masked_scce",
                 **kwargs):
        # ── PERBAIKAN: Matikan reduksi otomatis Keras di sini ──
        super().__init__(name=name, reduction=tf.keras.losses.Reduction.NONE, **kwargs)
        
        self.ignore_index       = ignore_index
        self.it_skill_label_ids = it_skill_label_ids or []
        self.skill_weight       = skill_weight
        self.label_smoothing    = label_smoothing

    def call(self, y_true, y_pred):
        """
        y_true : (batch, seq_len)  int32  — label ids, -100 = ignore
        y_pred : (batch, seq_len, num_labels)  float32 — logits
        """
        # Mask padding / sub-word continuations
        mask    = tf.cast(tf.not_equal(y_true, self.ignore_index), tf.float32)

        # Replace -100 with 0 so one_hot doesn't error; mask will zero it out later
        y_safe  = tf.where(tf.equal(y_true, self.ignore_index),
                           tf.zeros_like(y_true), y_true)

        # Konversi ke One-Hot agar mendukung label_smoothing
        num_classes = tf.shape(y_pred)[-1]
        y_safe_one_hot = tf.one_hot(y_safe, depth=num_classes)

        # Gunakan categorical_crossentropy standar
        per_tok = tf.keras.losses.categorical_crossentropy(
            y_safe_one_hot, y_pred, from_logits=True,
            label_smoothing=self.label_smoothing
        )   # (batch, seq_len)

        # Optional skill upweighting
        if self.it_skill_label_ids:
            it_ids  = tf.constant(self.it_skill_label_ids, dtype=tf.int32)
            is_skill = tf.reduce_any(
                tf.equal(tf.expand_dims(y_safe, -1),
                         tf.reshape(it_ids, [1, 1, -1])),
                axis=-1
            )   # (batch, seq_len)
            weights = tf.where(is_skill,
                               tf.fill(tf.shape(mask), self.skill_weight),
                               tf.ones_like(mask))
            per_tok = per_tok * weights

        # Apply mask and compute mean over valid tokens
        loss = tf.reduce_sum(per_tok * mask) / (tf.reduce_sum(mask) + 1e-8)
        return loss

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            "ignore_index"      : self.ignore_index,
            "it_skill_label_ids": self.it_skill_label_ids,
            "skill_weight"      : self.skill_weight,
            "label_smoothing"   : self.label_smoothing,
        })
        return cfg

# (Pastikan loss_fn di-generate ulang)
loss_fn = MaskedSparseCCELoss(
    ignore_index       = IGNORE_INDEX,
    it_skill_label_ids = IT_SKILL_LABEL_IDS,
    skill_weight       = 2.0,
    label_smoothing    = 0.05,
)

# Build skill label ids for upweighting
IT_SKILL_LABEL_IDS = [
    PRETRAINED_LABEL2ID[lbl]
    for lbl in PRETRAINED_LABEL2ID
    if any(sk in lbl for sk in IT_SKILL_ENTITY_TYPES)
]
print(f"IT skill label ids (upweighted in loss): {IT_SKILL_LABEL_IDS[:8]}…")

loss_fn = MaskedSparseCCELoss(
    ignore_index       = IGNORE_INDEX,
    it_skill_label_ids = IT_SKILL_LABEL_IDS,
    skill_weight       = 2.0,
    label_smoothing    = 0.05,
)
print("Custom loss function ready.")

In [ ]:
# ── (B) Custom Training Callback

class QLOPTrainingCallback:
    """
    Custom callback (non-Keras) used inside the manual GradientTape loop.
    Implements:
      • Epoch/step console logging with colour
      • Best-model checkpoint tracking
      • Early stopping on validation F1
    """

    RESET = "\033[0m"; BOLD = "\033[1m"; GREEN = "\033[92m"
    YELLOW = "\033[93m"; RED = "\033[91m"; CYAN = "\033[96m"

    def __init__(self, patience: int = 2, min_delta: float = 0.001):
        self.patience       = patience
        self.min_delta      = min_delta
        self.best_val_f1    = -np.inf
        self.wait           = 0
        self.stop_training  = False
        self.history        = {"train_loss": [], "val_loss": [],
                               "train_acc" : [], "val_acc" : [],
                               "val_f1"    : []}

    def on_epoch_begin(self, epoch: int, total_epochs: int):
        print(f"\n{self.BOLD}{self.CYAN}{'═'*60}")
        print(f"  Epoch {epoch+1}/{total_epochs}")
        print(f"{'═'*60}{self.RESET}")
        self._epoch_start = time.time()

    def on_step_end(self, step: int, total_steps: int, loss: float, step_interval: int = 50):
        if (step + 1) % step_interval == 0 or step == total_steps - 1:
            pct = (step + 1) / total_steps * 100
            bar = "▓" * int(pct / 5) + "░" * (20 - int(pct / 5))
            print(f"\r  [{bar}] {pct:5.1f}%  step={step+1}/{total_steps}  "
                  f"loss={loss:.4f}", end="", flush=True)

    def on_epoch_end(self, epoch: int, metrics: dict):
        elapsed = time.time() - self._epoch_start
        self.history["train_loss"].append(metrics.get("train_loss", 0))
        self.history["val_loss"  ].append(metrics.get("val_loss",   0))
        self.history["train_acc" ].append(metrics.get("train_acc",  0))
        self.history["val_acc"   ].append(metrics.get("val_acc",    0))
        self.history["val_f1"    ].append(metrics.get("val_f1",     0))

        val_f1 = metrics.get("val_f1", 0)
        improved = val_f1 > self.best_val_f1 + self.min_delta
        marker   = f"{self.GREEN}↑ NEW BEST{self.RESET}" if improved else ""

        print(f"\n  ⏱  {elapsed:.1f}s  "
              f"train_loss={metrics.get('train_loss',0):.4f}  "
              f"val_loss={metrics.get('val_loss',0):.4f}  "
              f"val_acc={metrics.get('val_acc',0):.4f}  "
              f"val_f1={val_f1:.4f}  {marker}")

        if improved:
            self.best_val_f1 = val_f1
            self.wait        = 0
        else:
            self.wait += 1
            print(f"  {self.YELLOW}⚠ No improvement for {self.wait}/{self.patience} epoch(s){self.RESET}")
            if self.wait >= self.patience:
                self.stop_training = True
                print(f"  {self.RED}🛑 Early stopping triggered!{self.RESET}")

    def on_train_end(self):
        print(f"\n{self.BOLD}Training complete — best val F1: {self.best_val_f1:.4f}{self.RESET}")


callback = QLOPTrainingCallback(patience=2)
print("Custom callback ready.")

##  Optimizer & Learning-Rate Schedule

In [ ]:
import tensorflow as tf

# ── Linear warmup + linear decay schedule 
total_steps   = len(train_tf) * CFG["epochs"]
warmup_steps  = int(total_steps * CFG["warmup_ratio"])
print(f"Total training steps : {total_steps}")
print(f"Warmup steps         : {warmup_steps}")

class LinearWarmupDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    """Linear warmup then linear decay to 0."""

    def __init__(self, peak_lr: float, warmup_steps: int, total_steps: int, **kw):
        super().__init__(**kw)
        self.peak_lr      = tf.cast(peak_lr,      tf.float32)
        self.warmup_steps = tf.cast(warmup_steps, tf.float32)
        self.total_steps  = tf.cast(total_steps,  tf.float32)

    def __call__(self, step):
        step  = tf.cast(step, tf.float32)
        warmup = self.peak_lr * (step / tf.maximum(self.warmup_steps, 1.0))
        decay  = self.peak_lr * tf.maximum(
            0.0,
            (self.total_steps - step) / tf.maximum(self.total_steps - self.warmup_steps, 1.0)
        )
        return tf.where(step < self.warmup_steps, warmup, decay)

    def get_config(self):
        return {"peak_lr": float(self.peak_lr.numpy()),
                "warmup_steps": int(self.warmup_steps.numpy()),
                "total_steps" : int(self.total_steps.numpy())}

with strategy.scope():
    lr_schedule = LinearWarmupDecay(
        peak_lr      = CFG["learning_rate"],
        warmup_steps = warmup_steps,
        total_steps  = total_steps,
    )
    
    # ── PERBAIKAN: Gunakan Adam standar alih-alih AdamW ──
    # Ini akan melewati (bypass) fungsi apply_weight_decay yang menyebabkan bug di TensorFlow
    optimizer = tf.keras.optimizers.Adam(
        learning_rate = lr_schedule,
        clipnorm      = 1.0,
        name          = "adam_qlop",
    )

print("✅ Optimizer Adam ready (Bug-free!).")

## TensorBoard Writers

In [ ]:
# ── TensorBoard summary writers 
TRAIN_LOG_DIR = os.path.join(CFG["log_dir"], "train",
                              datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
VAL_LOG_DIR   = os.path.join(CFG["log_dir"], "val",
                              datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))

train_writer = tf.summary.create_file_writer(TRAIN_LOG_DIR)
val_writer   = tf.summary.create_file_writer(VAL_LOG_DIR)

print(f"TensorBoard writers ready")
print(f"   Train log : {TRAIN_LOG_DIR}")
print(f"   Val   log : {VAL_LOG_DIR}")
print("\nTo launch TensorBoard in Kaggle:")
print("   %load_ext tensorboard")
print(f"   %tensorboard --logdir {CFG['log_dir']}")


## Metric Helper Functions

In [ ]:
# ── Metric helpers used inside the training loop

def compute_token_accuracy(logits, labels):
    """Token-level accuracy ignoring positions labelled -100."""
    preds = tf.argmax(logits, axis=-1, output_type=tf.int32)
    mask  = tf.cast(tf.not_equal(labels, IGNORE_INDEX), tf.float32)
    correct = tf.cast(tf.equal(preds, labels), tf.float32) * mask
    return float(tf.reduce_sum(correct) / (tf.reduce_sum(mask) + 1e-8))


def batch_predictions_to_seqeval(logits_list, labels_list):
    """
    Convert a list of (logits, labels) batches into seqeval-compatible lists
    of string-tag sequences (ignoring -100 positions).
    Returns (true_seqs, pred_seqs).
    """
    true_seqs, pred_seqs = [], []
    for logits, labels in zip(logits_list, labels_list):
        pred_ids  = tf.argmax(logits, axis=-1).numpy()   # (B, L)
        label_ids = labels.numpy()                        # (B, L)
        for pred_row, label_row in zip(pred_ids, label_ids):
            t_seq, p_seq = [], []
            for p, l in zip(pred_row, label_row):
                if l == IGNORE_INDEX:
                    continue
                t_seq.append(PRETRAINED_ID2LABEL.get(int(l),  "O"))
                p_seq.append(PRETRAINED_ID2LABEL.get(int(p),  "O"))
            true_seqs.append(t_seq)
            pred_seqs.append(p_seq)
    return true_seqs, pred_seqs


def evaluate_seqeval(logits_list, labels_list, verbose: bool = False):
    """Run seqeval F1 / precision / recall on collected batches."""
    true_seqs, pred_seqs = batch_predictions_to_seqeval(logits_list, labels_list)
    f1  = seq_f1(true_seqs,        pred_seqs, zero_division=0)
    pre = seq_precision(true_seqs, pred_seqs, zero_division=0)
    rec = seq_recall(true_seqs,    pred_seqs, zero_division=0)
    if verbose:
        print(classification_report(true_seqs, pred_seqs, zero_division=0))
    return {"f1": f1, "precision": pre, "recall": rec}

print("Metric helpers ready.")


In [ ]:
# ── PERSIAPAN FINAL SEBELUM TRAINING LOOP ──
import tensorflow as tf

print("Membersihkan sisa memori dari error sebelumnya...")
tf.keras.backend.clear_session()

with strategy.scope():
    print("Membangun ulang model & optimizer di dalam payung Multi-GPU...")
    
    # 1. Load Ulang Model Dasar
    _bert_base = TFBertForTokenClassification.from_pretrained(
        LOCAL_MODEL_DIR,
        from_pt=True,
        num_labels=NUM_LABELS
    )
    
    # 2. Bangun Custom Model QLOP
    model = QLOPNERModel(
        bert_model              = _bert_base,
        num_labels              = NUM_LABELS,
        dropout_rate            = CFG["dropout_rate"],
        unfreeze_last_n_layers  = 4,
        name                    = "qlop_ner",
    )
    
    # 3. Hubungkan ulang dengan Optimizer Adam
    optimizer = tf.keras.optimizers.Adam(
        learning_rate = lr_schedule,
        clipnorm      = 1.0,
        name          = "adam_qlop",
    )
    
    # 4. 🚨 TRICK MLOPS: "Warm-up" Inisialisasi 🚨
    print("Menyuapi dummy data untuk memaksa pembuatan bobot Dense Layer...")
    dummy_input = {
        "input_ids"     : tf.zeros([1, CFG["max_length"]], dtype=tf.int32),
        "attention_mask": tf.zeros([1, CFG["max_length"]], dtype=tf.int32),
        "token_type_ids": tf.zeros([1, CFG["max_length"]], dtype=tf.int32),
    }
    # Memaksa model melewati forward pass pertama dengan aman
    _ = model(dummy_input, training=False) 
    
    # Memaksa optimizer menyiapkan slot memori untuk variabel model
    optimizer.build(model.trainable_variables)
    
    print("✅ Arsitektur siap 100%! Tidak akan ada lazy initialization lagi.")

## Custom Training Loop (`tf.GradientTape`)  
This is the **critical** section implementing a full custom training + evaluation loop  
without `model.fit()`. TensorBoard metrics are written at every step and every epoch.


In [ ]:
# ── 1. Distribusikan Dataset ke Multi-GPU ──────────────────────────────────────
dist_train_tf = strategy.experimental_distribute_dataset(train_tf)
dist_val_tf   = strategy.experimental_distribute_dataset(val_tf)

# Helper: Menggabungkan tensor dari 2 GPU menjadi 1 tensor utuh untuk dihitung akurasinya
def unpack_distributed(tensor):
    if hasattr(tensor, 'values'):
        return tf.concat(tensor.values, axis=0)
    return tensor

# ── 2. Definisi Step Functions ─────────────────────────────────────────────────

# Catatan: Jangan pakai @tf.function di sini, melainkan di wrapper distribusinya
def train_step(inputs, labels):
    """One gradient update step per replica."""
    with tf.GradientTape() as tape:
        logits = model(inputs, training=True)
        loss   = loss_fn(labels, logits)
        # Scale loss for distributed training
        scaled_loss = loss / tf.cast(NUM_REPLICAS, tf.float32)

    grads = tape.gradient(scaled_loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss, logits

@tf.function(reduce_retracing=True)
def distributed_train_step(inputs, labels):
    # Mengeksekusi train_step secara paralel di semua GPU
    per_replica_losses, per_replica_logits = strategy.run(train_step, args=(inputs, labels))
    # Menjumlahkan loss dari semua GPU
    mean_loss = strategy.reduce(tf.distribute.ReduceOp.SUM, per_replica_losses, axis=None)
    return mean_loss, per_replica_logits, labels


def val_step(inputs, labels):
    """One inference step per replica."""
    logits = model(inputs, training=False)
    loss   = loss_fn(labels, logits)
    return loss, logits

@tf.function(reduce_retracing=True)
def distributed_val_step(inputs, labels):
    per_replica_losses, per_replica_logits = strategy.run(val_step, args=(inputs, labels))
    mean_loss = strategy.reduce(tf.distribute.ReduceOp.SUM, per_replica_losses, axis=None)
    return mean_loss, per_replica_logits, labels


# ── 3. Main Training Loop ──────────────────────────────────────────────────────

global_step = 0

for epoch in range(CFG["epochs"]):

    callback.on_epoch_begin(epoch, CFG["epochs"])

    # ── Training phase 
    epoch_train_loss = []
    epoch_train_acc  = []

    for step, (batch_inputs, batch_labels) in enumerate(dist_train_tf):
        # Jalankan eksekusi terdistribusi
        t_loss, t_logits, t_labels = distributed_train_step(batch_inputs, batch_labels)
        
        # Satukan data dari GPU 1 & GPU 2 agar bentuknya kembali normal (flat)
        t_logits_flat = unpack_distributed(t_logits)
        t_labels_flat = unpack_distributed(t_labels)

        t_acc            = compute_token_accuracy(t_logits_flat, t_labels_flat)
        t_loss_val       = float(t_loss)

        epoch_train_loss.append(t_loss_val)
        epoch_train_acc .append(t_acc)

        # # ── TensorBoard: per-step logging 
        # with train_writer.as_default():
        #     tf.summary.scalar("step/loss",         t_loss_val, step=global_step)
        #     tf.summary.scalar("step/accuracy",     t_acc,      step=global_step)
        #     tf.summary.scalar("step/learning_rate",
        #                        float(optimizer.learning_rate(global_step)),
        #                        step=global_step)
        # ── TensorBoard: per-step logging 
        with train_writer.as_default():
            tf.summary.scalar("step/loss",         t_loss_val, step=global_step)
            tf.summary.scalar("step/accuracy",     t_acc,      step=global_step)
            
            # ── PERBAIKAN: Panggil lr_schedule secara langsung ──
            tf.summary.scalar("step/learning_rate",
                               float(lr_schedule(global_step)),
                               step=global_step)

        callback.on_step_end(step, len(train_tf), t_loss_val)
        global_step += 1

    mean_train_loss = float(np.mean(epoch_train_loss))
    mean_train_acc  = float(np.mean(epoch_train_acc))

    # ── Validation phase 
    epoch_val_loss   = []
    epoch_val_acc    = []
    val_logits_list  = []
    val_labels_list  = []

    for batch_inputs, batch_labels in dist_val_tf:
        v_loss, v_logits, v_labels = distributed_val_step(batch_inputs, batch_labels)
        
        # Satukan data dari multi-GPU
        v_logits_flat = unpack_distributed(v_logits)
        v_labels_flat = unpack_distributed(v_labels)

        v_acc = compute_token_accuracy(v_logits_flat, v_labels_flat)
        
        epoch_val_loss.append(float(v_loss))
        epoch_val_acc .append(v_acc)
        val_logits_list.append(v_logits_flat)
        val_labels_list.append(v_labels_flat)

    mean_val_loss = float(np.mean(epoch_val_loss))
    mean_val_acc  = float(np.mean(epoch_val_acc))

    # seqeval F1 on validation set
    seqeval_metrics = evaluate_seqeval(val_logits_list, val_labels_list)
    val_f1 = seqeval_metrics["f1"]

    # ── TensorBoard: per-epoch logging 
    with train_writer.as_default():
        tf.summary.scalar("epoch/loss",     mean_train_loss, step=epoch)
        tf.summary.scalar("epoch/accuracy", mean_train_acc,  step=epoch)

    with val_writer.as_default():
        tf.summary.scalar("epoch/loss",      mean_val_loss,                 step=epoch)
        tf.summary.scalar("epoch/accuracy",  mean_val_acc,                  step=epoch)
        tf.summary.scalar("epoch/f1",        val_f1,                        step=epoch)
        tf.summary.scalar("epoch/precision", seqeval_metrics["precision"],  step=epoch)
        tf.summary.scalar("epoch/recall",    seqeval_metrics["recall"],     step=epoch)

    train_writer.flush()
    val_writer.flush()

    # ── Callback on_epoch_end 
    callback.on_epoch_end(epoch, {
        "train_loss": mean_train_loss,
        "val_loss"  : mean_val_loss,
        "train_acc" : mean_train_acc,
        "val_acc"   : mean_val_acc,
        "val_f1"    : val_f1,
    })

    # ── Save best checkpoint 
    import os
    if val_f1 >= callback.best_val_f1:
        # Menyimpan bobot di Kaggle local working directory
        model.save_weights("/kaggle/working/best_weights.h5")
        print(f"  Best weights saved (val_f1={val_f1:.4f})")

    if callback.stop_training:
        break

callback.on_train_end()

## Training History Visualisation

In [ ]:
history = callback.history
epochs_ran = len(history["train_loss"])

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Loss
axes[0].plot(range(1, epochs_ran+1), history["train_loss"], "o-", label="Train",  color="#4A90D9")
axes[0].plot(range(1, epochs_ran+1), history["val_loss"],   "s--", label="Val",   color="#E8453C")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(range(1, epochs_ran+1), history["train_acc"], "o-", label="Train", color="#4A90D9")
axes[1].plot(range(1, epochs_ran+1), history["val_acc"],   "s--", label="Val",  color="#E8453C")
axes[1].axhline(0.85, color="green", ls=":", lw=2, label="Target 85%")
axes[1].set_title("Token Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

# F1
axes[2].plot(range(1, epochs_ran+1), history["val_f1"], "D-", color="#7B68EE", lw=2)
axes[2].axhline(0.85, color="green", ls=":", lw=2, label="Target 85%")
axes[2].set_title("Val seqeval F1"); axes[2].set_xlabel("Epoch"); axes[2].legend(); axes[2].grid(True, alpha=0.3)
axes[2].set_ylim(0, 1.0)

plt.suptitle("QLOP NER — Training History", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/kaggle/working/training_history.png", dpi=120, bbox_inches="tight")
plt.show()
print("✅ Training history saved.")


## Full Test-Set Evaluation

In [ ]:
# ── Load best weights before evaluating
best_weights_path = os.path.join(CFG["model_save_path"], "best_weights.h5")
if os.path.exists(best_weights_path):
    model.load_weights(best_weights_path)
    print("Best weights loaded for evaluation.")
else:
    print("  Best weights not found — evaluating with current weights.")

# ── Collect test predictions 
test_logits_list = []
test_labels_list = []
test_loss_acc    = []

for batch_inputs, batch_labels in test_tf:
    t_loss, t_logits = val_step(batch_inputs, batch_labels)
    t_acc            = compute_token_accuracy(t_logits, batch_labels)
    test_logits_list.append(t_logits)
    test_labels_list.append(batch_labels)
    test_loss_acc.append((float(t_loss), t_acc))

mean_test_loss = np.mean([x[0] for x in test_loss_acc])
mean_test_acc  = np.mean([x[1] for x in test_loss_acc])

# ── seqeval metrics 
print("\n" + "═"*60)
print("  FULL CLASSIFICATION REPORT (seqeval)")
print("═"*60)
test_metrics = evaluate_seqeval(test_logits_list, test_labels_list, verbose=True)

print(f"\nSummary Metrics")
print(f"  Test Loss      : {mean_test_loss:.4f}")
print(f"  Token Accuracy : {mean_test_acc:.4f}  {'✅' if mean_test_acc >= 0.85 else '❌'} (target ≥ 0.85)")
print(f"  seqeval F1     : {test_metrics['f1']:.4f}  {'✅' if test_metrics['f1'] >= 0.85 else '❌'} (target ≥ 0.85)")
print(f"  seqeval Prec   : {test_metrics['precision']:.4f}")
print(f"  seqeval Recall : {test_metrics['recall']:.4f}")


## Model Export (Production-Ready)

In [ ]:
# ── 17.1  SavedModel format 
print("Saving as SavedModel…")
# We need to build the model with a concrete input before saving
dummy_input = {
    "input_ids"     : tf.zeros((1, CFG["max_length"]), dtype=tf.int32),
    "attention_mask": tf.ones ((1, CFG["max_length"]), dtype=tf.int32),
    "token_type_ids": tf.zeros((1, CFG["max_length"]), dtype=tf.int32),
}
_ = model(dummy_input, training=False)  # ensure model is built

tf.saved_model.save(model, CFG["model_save_path"])
print(f" SavedModel saved → {CFG['model_save_path']}")


In [ ]:
# ── 17.2  .keras format 
# For .keras we save only the weights (full model save of custom subclass
# requires Keras serialisation registrations).
model.save_weights(CFG["keras_save_path"].replace(".keras", "_final.weights.h5"))
print(f" Weights saved  → {CFG['keras_save_path'].replace('.keras', '_final.weights.h5')}")

# ── 17.3  Export tokenizer & label config alongside model 
tokenizer.save_pretrained(os.path.join(CFG["model_save_path"], "tokenizer"))

model_meta = {
    "model_name"         : "QLOP NER v1",
    "base_model"         : CFG["base_model"],
    "num_labels"         : NUM_LABELS,
    "id2label"           : {str(k): v for k, v in PRETRAINED_ID2LABEL.items()},
    "label2id"           : {v: str(k) for k, v in PRETRAINED_ID2LABEL.items()},
    "max_length"         : CFG["max_length"],
    "test_f1"            : round(test_metrics["f1"], 4),
    "test_accuracy"      : round(float(mean_test_acc), 4),
    "it_skill_entities"  : list(IT_SKILL_ENTITY_TYPES),
}

with open(os.path.join(CFG["model_save_path"], "model_meta.json"), "w") as f:
    json.dump(model_meta, f, indent=2)

print(" Tokenizer + model metadata exported.")
print("\n Exported artifacts:")
for root, dirs, files in os.walk(CFG["model_save_path"]):
    for fname in files:
        fpath = os.path.join(root, fname)
        size  = os.path.getsize(fpath) / 1e6
        print(f"  {fpath}  ({size:.1f} MB)")


## Inference Script  
Clean, production-ready entity extraction from raw resume text.  
**IT Hard-Skill entities are highlighted.**


In [ ]:
# ── Inference helper ─────────────────────────────────────────────────────────

def extract_it_entities(
    text             : str,
    conf_threshold   : float = 0.5,
    it_focus_only    : bool  = False,
) -> list[dict]:
    """
    Extract named entities from a raw resume string.

    Args:
        text           : Raw resume text (or sentence).
        conf_threshold : Minimum softmax confidence to include an entity.
        it_focus_only  : If True, return only IT hard-skill entity types.

    Returns:
        List of dicts: [{text, label, confidence, start_token, end_token}]
    """
    # Tokenise
    encoding = tokenizer(
        text,
        truncation     = True,
        max_length     = CFG["max_length"],
        padding        = "max_length",
        return_tensors = "tf",
    )
    inputs = {
        "input_ids"     : encoding["input_ids"],
        "attention_mask": encoding["attention_mask"],
        "token_type_ids": encoding.get("token_type_ids",
                                        tf.zeros_like(encoding["input_ids"])),
    }

    # Predict
    logits = model(inputs, training=False)            # (1, seq_len, num_labels)
    probs  = tf.nn.softmax(logits, axis=-1)[0]        # (seq_len, num_labels)
    pred_ids   = tf.argmax(probs, axis=-1).numpy()    # (seq_len,)
    confidences= tf.reduce_max(probs, axis=-1).numpy()# (seq_len,)

    # Decode tokens and align back to words (sub-word merging)
    tokens     = tokenizer.convert_ids_to_tokens(encoding["input_ids"][0].numpy())
    word_ids   = encoding.word_ids()   # None for special tokens

    entities   = []
    current    = None

    for idx, (tok, pred_id, conf, word_id) in enumerate(
            zip(tokens, pred_ids, confidences, word_ids)):

        if word_id is None:
            # Special token — close any open entity
            if current:
                entities.append(current)
                current = None
            continue

        label = PRETRAINED_ID2LABEL.get(int(pred_id), "O")

        if label.startswith("B-"):
            if current:
                entities.append(current)
            entity_type = label[2:]
            current = {
                "text"       : tok.replace("##", ""),
                "label"      : entity_type,
                "confidence" : float(conf),
                "start_token": idx,
                "end_token"  : idx,
            }

        elif label.startswith("I-") and current:
            entity_type = label[2:]
            if entity_type == current["label"]:
                piece = tok.replace("##", "")
                # Sub-word pieces start with ##; merge without space
                if tok.startswith("##"):
                    current["text"] += piece
                else:
                    current["text"] += " " + piece
                current["end_token"]   = idx
                current["confidence"]  = min(current["confidence"], float(conf))

        else:
            if current:
                entities.append(current)
                current = None

    if current:
        entities.append(current)

    # Filter by confidence & optionally by IT scope
    entities = [e for e in entities if e["confidence"] >= conf_threshold]
    if it_focus_only:
        entities = [e for e in entities if e["label"] in IT_SKILL_ENTITY_TYPES]

    return entities


def print_entities(text: str, entities: list[dict], title: str = "Extracted Entities"):
    """Pretty-print extracted entities with colour highlighting."""
    IT_COLOUR  = "\033[92m"   # green
    GEN_COLOUR = "\033[94m"   # blue
    RESET      = "\033[0m"
    BOLD       = "\033[1m"

    print(f"\n{BOLD}{'─'*60}")
    print(f"  {title}")
    print(f"{'─'*60}{RESET}")
    print(f"  Input: {text[:120]}{'…' if len(text)>120 else ''}")
    print(f"{'─'*60}")

    if not entities:
        print("  (no entities found)")
        return

    for e in entities:
        colour = IT_COLOUR if e["label"] in IT_SKILL_ENTITY_TYPES else GEN_COLOUR
        tag    = "🎯" if e["label"] in IT_SKILL_ENTITY_TYPES else "  "
        print(f"  {tag} {colour}{e['label']:30s}{RESET}  "
              f"'{BOLD}{e['text']}{RESET}'  (conf={e['confidence']:.3f})")

    print(f"{'─'*60}")
    it_count  = sum(1 for e in entities if e["label"] in IT_SKILL_ENTITY_TYPES)
    gen_count = len(entities) - it_count
    print(f"  Total: {len(entities)} entities  |  "
          f"🎯 IT-skill: {it_count}  |  general: {gen_count}")


print(" Inference helpers ready.")


In [ ]:
# ── Run inference on sample IT resume snippets

TEST_RESUMES = [
    # 1 ── Full-stack developer
    ("I am a Full Stack Developer with 5 years of experience specialising in React, "
     "Node.js, and TypeScript. Graduated with a B.Tech in Computer Science from IIT "
     "Delhi in 2019. Previously worked at Infosys and currently at Razorpay."),

    # 2 ── Data scientist
    ("Sarah Johnson is a Senior Data Scientist at Google with 8 years of experience. "
     "She holds an M.Sc in Data Science from Stanford University (2016). "
     "Core skills: Python, TensorFlow, PyTorch, Spark, SQL, Tableau."),

    # 3 ── DevOps / Cloud engineer
    ("DevOps Engineer with 6 years of experience in AWS, GCP, Kubernetes, Terraform, "
     "and CI/CD pipelines. B.E. Information Technology from NIT Trichy (2018). "
     "Contact: devops.raj@gmail.com  |  Bangalore"),

    # 4 ── ML Engineer (abbreviated)
    "ML Engineer  Python  PyTorch  Docker  Kubernetes  3 years at Amazon",

    # 5 ── Raw skill dump
    "Skills: Java, Spring Boot, Microservices, PostgreSQL, Redis, Kafka, REST API, Git",
]

for i, resume_text in enumerate(TEST_RESUMES, 1):
    entities = extract_it_entities(resume_text, conf_threshold=0.4, it_focus_only=False)
    print_entities(resume_text, entities, title=f"Resume Sample #{i}")


## Batch Inference & Structured Output

In [ ]:
import csv

def batch_extract_to_csv(texts: list[str], output_path: str,
                          conf_threshold: float = 0.4) -> None:
    """Extract IT entities from multiple resume snippets and save to CSV."""
    rows = []
    for idx, text in enumerate(texts):
        entities = extract_it_entities(text, conf_threshold=conf_threshold,
                                        it_focus_only=True)
        for e in entities:
            rows.append({
                "resume_id"  : idx + 1,
                "entity_text": e["text"],
                "entity_type": e["label"],
                "confidence" : round(e["confidence"], 3),
            })
        if not entities:
            rows.append({"resume_id": idx + 1, "entity_text": "",
                         "entity_type": "NONE", "confidence": 0.0})

    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["resume_id","entity_text",
                                                "entity_type","confidence"])
        writer.writeheader()
        writer.writerows(rows)
    print(f"✅ {len(rows)} rows exported → {output_path}")


batch_extract_to_csv(
    TEST_RESUMES,
    output_path="/kaggle/working/extracted_entities.csv",
)

# Preview CSV
import pandas as pd
df_out = pd.read_csv("/kaggle/working/extracted_entities.csv")
print("\nExtracted IT entities (preview):")
print(df_out.to_string(index=False))


## TensorBoard & Final Summary

In [ ]:
# ── Launch TensorBoard (works in Kaggle / Colab) 
try:
    get_ipython                               # only in notebook context
    from IPython import get_ipython
    get_ipython().run_line_magic("load_ext", "tensorboard")
    get_ipython().run_line_magic("tensorboard", f"--logdir {CFG['log_dir']}")
except Exception:
    print(f"Run manually:  tensorboard --logdir {CFG['log_dir']}")


In [ ]:
# ── Final project summary
print("\n" + "═"*65)
print("  QLOP — NER Model Training Summary")
print("═"*65)
print(f"  Model          : {CFG['base_model']} (PyTorch → TensorFlow)")
print(f"  Architecture   : TFBertForTokenClassification + ITSkillAttentionHead")
print(f"  Epochs trained : {len(callback.history['train_loss'])}")
print(f"  Best Val F1    : {callback.best_val_f1:.4f}")
print(f"  Test Token Acc : {mean_test_acc:.4f}  {'✅ PASS' if mean_test_acc >= 0.85 else '❌ FAIL'}")
print(f"  Test seqeval F1: {test_metrics['f1']:.4f}  {'✅ PASS' if test_metrics['f1'] >= 0.85 else '❌ FAIL'}")
print("─"*65)
print("  Main Quest Checklist:")
checks = [
    ("TF Porting (from_pt=True)",              True),
    ("Functional API / Model Subclassing",      True),
    ("Heavy Fine-Tuning (top 4 BERT layers)",   True),
    ("Custom Loss (MaskedSparseCCELoss)",        True),
    ("Custom Callback (QLOPTrainingCallback)",   True),
    ("Custom Training Loop (GradientTape)",      True),
    ("TensorBoard Integration",                  True),
    ("Evaluation Metrics (seqeval F1)",          True),
    ("Model Export (SavedModel + weights)",      True),
    ("Inference Script",                         True),
]
for desc, done in checks:
    status = "✅" if done else "❌"
    print(f"    {status}  {desc}")
print("═"*65)
